Two implementation of computing the sum of squares for 1 million numbers:

In [2]:
import numpy as np 
arr = np.arange(1_000_000, dtype=np.float64)


In [7]:
# Write the loop version using a Python for loop.
def square_sum_python_loop(arr):
    total=0.0
    for x in arr:
        total+=x*x
    return total
# Write the vectorized version using NumPy.
def square_sum_vectorized(arr):
    return np.sum(arr**2)


In [11]:
# Time both using %timeit and report the seedup factor.
%timeit square_sum_python_loop(arr)
%timeit square_sum_vectorized(arr)

107 ms ± 2.44 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
3.69 ms ± 300 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
# Now repeat with dtype=np.float32 on the vectorized version - does dtype affect speed?
arr2 = np.arange(1_000_000, dtype=np.float32)
%timeit square_sum_vectorized(arr2)

1.77 ms ± 109 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Yes, dtype affect speed. Higher the dtype size higher will be the time consumption.

Given this loop-base implementation:

In [22]:
np.random.seed(0)
data = np.random.randint(0, 100, size=(1000, 500))

def python_loop(data):
    result = []
    for row in data:
        count = 0
        for val in row:
            if val > 50:
                count += 1
        result.append(count)
    result = np.array(result)

In [16]:
# Rewrite this entirely without any Python loops - single line preferred.
def numpy_vectorized(data):
    return np.sum(data>50,axis=1)

In [18]:
# Time both version.
%timeit python_loop(data)
%timeit numpy_vectorized(data)

34.7 ms ± 1.54 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
454 μs ± 24.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
# Verify ouput match.
print("Matched?: ",np.array_equal(numpy_vectorized(data),result))

Matched?:  True


A large array and need to apply this piecewise logic:

In [57]:
arr = np.random.randint(-100, 100, size=2_000_000)

# Logic:
# if value < -50  → set to -50
# if value > 50   → set to 50
# else            → keep as is
print("Original sum: ",arr.sum())

Original sum:  -952307


In [67]:
# Write it first using np.vectorize() with a Python function.
import time as time
def manipulate(value):
    if value<-50:
        return -50
    elif value>50:
        return 50
    else:
        return value
fstart=time.time()
result = np.vectorize(manipulate)(arr)
fend=time.time()
print("Result sum: ",result.sum())

Result sum:  -481285


In [68]:
# Write the proper vectorized verson without np.vectorize()
vstart=time.time()
vresult=np.where(arr>50,50,np.where(arr<-50,-50,arr))
vend=time.time()
print("VResult sum: ",vresult.sum())

VResult sum:  -481285


In [69]:
# Time both - explain why one is faster despite both begin "NumPy".
print("Function vector time: ",fend-fstart)
print("Proper vecorized time: ",vend-vstart)


Function vector time:  0.31928372383117676
Proper vecorized time:  0.010689258575439453


Function version is slower because it is still uses python's loop under the hood. np.vectorize() just call the function iteratively which is python loop. Other version is faster because it uses numpy.

A (5000,4) dataset of sensor readings:

In [3]:
np.random.seed(1)
sensors = np.random.normal(50, 15, size=(5000, 4)).astype(np.float64)

In [4]:
# Compute row-wise mean using a Python loop over rows vs np.mean(axis=1). Time both approaches
def row_wise_mean(data):
    sum=0
    for i in data:
        sum+=i
    return sum/len(data)
def vectorized_mean(data):
    return np.mean(data,axis=1)

%timeit row_wise_mean(sensors)
%timeit vectorized_mean(sensors)

2.96 ms ± 208 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
73.8 μs ± 2.7 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [11]:
# Compare memory: convert sensors to float32 using astype(copy=False).
small_sensors=sensors.astype(np.float32,copy=False)
print("Original sensor memory use: ",sensors.nbytes)
print("Reduced sensor memory use: ",small_sensors.nbytes)
print("Memory saved: ",sensors.nbytes-small_sensors.nbytes)

Original sensor memory use:  160000
Reduced sensor memory use:  80000
Memory saved:  80000
